# Grouped Query Attention (GQA)

GQA 是 MHA 和 MQA 之间的折中方案：

| | MHA | GQA | MQA |
|---|---|---|---|
| Q heads | H | H | H |
| K/V heads | H | G（G < H） | 1 |
| 参数量 | 最多 | 中等 | 最少 |
| 推理速度 | 最慢 | 快 | 最快 |
| 效果 | 最好 | 接近 MHA | 略差 |

**核心思想**：Q 保留 H 个 head，K/V 只有 G 组，每组 K/V 被 H/G 个 Q head 共享。

LLaMA 3、Mistral、Gemma 等主流 LLM 都使用 GQA。

## 为什么 K/V 可以共享？

推理时 KV Cache 是内存瓶颈：每个 token 都要缓存所有 head 的 K/V，MHA 的 KV Cache 大小正比于 H。

GQA 把 K/V head 数从 H 减到 G，KV Cache 缩小 H/G 倍，推理速度大幅提升，而效果损失很小。

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        assert num_heads % num_kv_heads == 0  # Q heads 必须能被 KV heads 整除
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)                       # Q: 全部 H 个 head
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)       # K: 只有 G 个 head
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)       # V: 只有 G 个 head
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape

        q = self.W_q(x).view(B, S, self.num_heads,    self.d_k).transpose(1, 2)  # (B, H, S, d_k)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)  # (B, G, S, d_k)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)  # (B, G, S, d_k)

        # 每组 K/V 复制 H/G 次，扩展到和 Q 相同的 head 数
        repeats = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(repeats, dim=1)  # (B, H, S, d_k)
        v = v.repeat_interleave(repeats, dim=1)  # (B, H, S, d_k)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, H, S, S)
        weights = torch.softmax(scores, dim=-1)
        attn = weights @ v                                        # (B, H, S, d_k)

        out = attn.transpose(1, 2).contiguous().view(B, S, -1)   # (B, S, d_model)
        return self.W_o(out)

## repeat_interleave vs repeat

这是 GQA 实现的关键，两者效果不同：

```python
# 假设 G=2，H=4，repeats=2
# k 的 head 顺序是 [0, 1]

k.repeat_interleave(2, dim=1)  # [0, 0, 1, 1]  ← 每个 head 紧挨着复制
k.repeat(1, 2, 1, 1)           # [0, 1, 0, 1]  ← 整体重复
```

`repeat_interleave` 让相邻的 Q head 共享同一组 K/V，分组关系更自然。

## GQA vs MHA：关键区别

GQA 只改了两处，其余和 MHA 完全一样：

**1. W_k / W_v 输出维度用 G 而不是 H**

```python
# MHA
self.W_k = nn.Linear(d_model, num_heads * self.d_k)     # H 个 head

# GQA
self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)  # G 个 head，G < H
```

**2. 多了 repeat_interleave 把 K/V 从 G 扩展到 H**

```python
k = k.repeat_interleave(repeats, dim=1)  # (B, G, S, d_k) -> (B, H, S, d_k)
v = v.repeat_interleave(repeats, dim=1)
```

之后的 attention 计算和 MHA 完全一样。本质上就是 K/V 少算了，再用复制凑齐 head 数。

## 验证

In [ ]:
# d_model=32, Q heads=8, KV heads=2 → 每组 KV 被 4 个 Q head 共享
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
x = torch.randn(1, 4, 32)  # (batch=1, seq_len=4, d_model=32)
print('Output:', gqa.forward(x).shape)  # 应为 (1, 4, 32)